# Orbital Ambulance

A **30-level** space ambulance supply game for AI Fundamentals topic 14. Choose a fixed floor size (**5×5**, **10×10**, **20×20**, **40×40**, **60×60**, **100×100**) or **Random**. **You drive the ambulance**; a new station map and visual look follow each delivery. Random size never repeats immediately.

**Course methods:** BFS finds the route, Bayes updates the free-hallway probability, HMM/Viterbi interprets three reports, and MDP advises Continue or Pause. **You decide whether to move.** The assigned topic requires BFS, Bayes and MDP; HMM is an extra demonstration. All maps and probabilities are simulated. No patient data is used.

## How to play in Colab

1. Click **Runtime → Run all** and wait for the game below.
2. Choose **Map size**, then click **Ready · start mission**. For a first look, choose **5**.
3. Solve the BFS challenge, then close every **Signal Interference** tab before driving. Level 1 has five tabs; each later level adds one. Bayes and HMM challenges appear along the route; an MDP challenge appears before the goal. A correct answer reveals its calculation and moves the ambulance one step.
4. **Scan** checks the next corridor, **Wait** allows another sensor reading, and **Hint** reveals the BFS route at a cost of half the level's starting energy. The hint starts off. A warning changes the energy cost of moving but never locks the controls.
5. Click **Pause / Resume** to stop or continue. **Menu** returns to the size chooser. The next level starts automatically after you reach the ward.
6. On large floors the view follows the ambulance. All 30 levels are generated by Python; every level selects a random look.

Use the **Background music** player inside the notebook; browser autoplay may be blocked. The desktop `.py` version starts its music automatically and provides the larger map view and vector character designs.

## Python algorithms and level generator

These functions are executed by the game. The map generator checks that both wards are reachable. BFS, Bayes, Viterbi and MDP are implemented in Python, not just printed as labels.

In [ ]:
from collections import deque
from dataclasses import dataclass
import colorsys
import random

TOTAL_LEVELS = 30
SIZE_CHOICES = (5, 10, 20, 40, 60, 100)
STYLE_NAMES = ("Clinic", "Pixel", "Realistic", "Mixed")
CHARACTERS = ("Space Ambulance",)
THEME_NAMES = [f"{i:03d} {STYLE_NAMES[min((i - 1)//40, 3)]}"
               for i in range(1, 159)]
HMM_TRANSITION = {
    "free": {"free": .80, "blocked": .20},
    "blocked": {"free": .30, "blocked": .70},
}
REWARDS = {
    "CONTINUE": {"free": 10, "blocked": -20},
    "PAUSE": {"free": -2, "blocked": 2},
}


def bfs_trace(start, goal, size, walls):
    """Return a shortest four-neighbor route and number of expanded cells."""
    queue = deque([start])
    parent = {start: None}
    expanded = 0
    while queue:
        cell = queue.popleft()
        expanded += 1
        if cell == goal:
            route = []
            while cell is not None:
                route.append(cell)
                cell = parent[cell]
            return route[::-1], expanded
        x, y = cell
        for next_cell in ((x + 1, y), (x, y - 1), (x - 1, y), (x, y + 1)):
            nx, ny = next_cell
            if (0 <= nx < size and 0 <= ny < size and next_cell not in walls
                    and next_cell not in parent):
                parent[next_cell] = cell
                queue.append(next_cell)
    return [], expanded


def bfs(start, goal, size, walls):
    return bfs_trace(start, goal, size, walls)[0]


def bayes_free(report, prior, sensitivity, false_alarm):
    """P(free | one CLEAR or ALERT report)."""
    if report not in ("CLEAR", "ALERT"):
        raise ValueError("report must be CLEAR or ALERT")
    p_report_free = sensitivity if report == "CLEAR" else 1 - sensitivity
    p_report_blocked = false_alarm if report == "CLEAR" else 1 - false_alarm
    free_part = prior * p_report_free
    blocked_part = (1 - prior) * p_report_blocked
    return free_part / (free_part + blocked_part), free_part, blocked_part


def viterbi(observations, prior, sensitivity, false_alarm):
    """Most likely sequence of the hidden free/blocked corridor states."""
    if not observations or any(x not in ("CLEAR", "ALERT") for x in observations):
        raise ValueError("observations must contain CLEAR or ALERT")
    states = ("free", "blocked")

    def emission(state, report):
        p_clear = sensitivity if state == "free" else false_alarm
        return p_clear if report == "CLEAR" else 1 - p_clear

    values = {"free": prior * emission("free", observations[0]),
              "blocked": (1 - prior) * emission("blocked", observations[0])}
    paths = {state: (state,) for state in states}
    trace = [values.copy()]
    for report in observations[1:]:
        next_values, next_paths = {}, {}
        for state in states:
            previous = max(states, key=lambda old: values[old] * HMM_TRANSITION[old][state])
            next_values[state] = (values[previous] * HMM_TRANSITION[previous][state]
                                  * emission(state, report))
            next_paths[state] = paths[previous] + (state,)
        values, paths = next_values, next_paths
        trace.append(values.copy())
    final = max(states, key=values.get)
    return paths[final], trace


def mdp_decision(prob_free):
    """Compare one-step expected rewards in two states for two actions."""
    q = {action: prob_free * reward["free"] + (1 - prob_free) * reward["blocked"]
         for action, reward in REWARDS.items()}
    return max(q, key=q.get), q


def level_size(mode, level_number, campaign_seed):
    """Fixed size, or a fresh random size that differs from the previous level."""
    if mode != "Random":
        return int(mode)
    rng = random.Random(campaign_seed)
    previous = None
    selected = SIZE_CHOICES[0]
    for _ in range(level_number):
        selected = rng.choice([size for size in SIZE_CHOICES if size != previous])
        previous = selected
    return selected


@dataclass(frozen=True)
class Floor:
    size: int
    start: tuple
    wards: dict
    walls: frozenset
    prior: float
    sensitivity: float
    false_alarm: float
    level_number: int


def generate_floor(size, level_number, seed):
    """Generate a solvable level with two wards and increasing wall density."""
    if size not in SIZE_CHOICES or not 1 <= level_number <= TOTAL_LEVELS:
        raise ValueError("unsupported size or level")
    start = (0, size // 2)
    wards = {"A": (size - 1, 0), "B": (size - 1, size - 1)}
    reserved = {start, *wards.values()}
    candidates = [(x, y) for y in range(size) for x in range(size)
                  if (x, y) not in reserved]
    density = min(.13 + .0065 * (level_number - 1), .33)
    wall_count = round(len(candidates) * density)
    rng = random.Random(seed)
    for _ in range(160):
        if size == 5:
            walls = frozenset(rng.sample(candidates, wall_count))
        else:
            grouped = set()
            target_grouped = round(wall_count * .65)
            max_span = max(2, min(7, size // 5))
            attempts = 0
            while len(grouped) < target_grouped and attempts < 2000:
                attempts += 1
                x0 = rng.randrange(1, size - 1)
                y0 = rng.randrange(1, size - 1)
                width = rng.randint(2, max_span)
                height = rng.randint(2, max_span)
                for y in range(y0, min(size - 1, y0 + height)):
                    for x in range(x0, min(size - 1, x0 + width)):
                        if len(grouped) >= target_grouped:
                            break
                        if (x, y) not in reserved:
                            grouped.add((x, y))
            remaining = [cell for cell in candidates if cell not in grouped]
            walls = frozenset(grouped | set(rng.sample(remaining, wall_count - len(grouped))))
        if all(bfs(start, ward, size, walls) for ward in wards.values()):
            break
    else:
        # A deterministic clear corridor guarantees a playable fallback.
        walls = set(rng.sample(candidates, wall_count))
        for x in range(size):
            walls.discard((x, start[1]))
        for y in range(size):
            walls.discard((size - 1, y))
        walls = frozenset(walls)
    prior = max(.54, .82 - .008 * (level_number - 1))
    sensitivity = max(.78, .94 - .005 * (level_number - 1))
    false_alarm = min(.28, .12 + .005 * (level_number - 1))
    return Floor(size, start, wards, walls, prior, sensitivity, false_alarm, level_number)


def theme_colors(index):
    """A palette for one of 158 generated looks in four map styles."""
    if not 1 <= index <= 158:
        raise ValueError("theme index must be 1..158")
    family = min((index - 1) // 40, 3)
    variant = (index - 1) % 40
    # Dark, low-saturation palettes only. Variants stay comfortable to view.
    hues = (.43, .51, .58, .34, .10, .68, .03, .47)
    hue = hues[variant % len(hues)]

    def color(saturation, value, h=hue):
        return "#%02x%02x%02x" % tuple(round(c * 255) for c in colorsys.hsv_to_rgb(h, saturation, value))

    # All four families use a readable, restrained UI; the map styling changes.
    return {
        "background": "#12191b",
        "card": "#1c2729",
        "ink": "#e1e8e4",
        "muted": "#a7b5b0",
        "line": "#344245",
        "accent": color(.23, .59 + index * .00065),
        "floor": "#263335" if family in (0, 3) else "#293639",
        "wall": "#526064" if family != 2 else "#596366",
        "path": color(.24, .43),
        "highlight": "#31423f",
        "family": family,
    }

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ipywidgets"])
    import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import asyncio
import math
MUSIC_DATA = '...'

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

class ColabCampaign:
    ROBOT_SYMBOLS = {"Space Ambulance":"✚"}

    def __init__(self):
        self.seed = random.SystemRandom().randrange(10**9)
        self.rng = random.Random(self.seed)
        self.mode = "Random"
        self.level_number = 0
        self.floor = None
        self.target = "A"
        self.route = []
        self.expanded = 0
        self.player_position = (0,0)
        self.moves = 0
        self.energy = 0
        self.score = 0
        self.facing = (1,0)
        self.scanned_cell = None
        self.hazard_state = None
        self.show_hint = False
        self.hidden_state = "free"
        self.reports = []
        self.bayes_result = None
        self.hmm_result = None
        self.mdp_result = None
        self.theme_index = 1
        self.robot_name = "Space Ambulance"
        self.ads_total = 0
        self.ads_remaining = 0
        self.ads_pending = False
        self.state = "menu"
        self.paused = False
        self.challenge_kind = None
        self.challenge_solved = set()
        self.challenge_answer = None
        self.challenge_explanation = ""
        self.challenge_result = False
        self.challenge_pending = None
        self.challenge_message = ""
        self.session = 0
        self.events = deque(maxlen=2)
        self.size_menu = widgets.Dropdown(options=["Random","5","10","20","40","60","100"],
                                          value="5", description="Map size:",
                                          layout=widgets.Layout(width="270px"))
        self.sound_toggle = widgets.Checkbox(value=False, description="Sound")
        self.start_button = widgets.Button(description="Ready · start mission",
                                           layout=widgets.Layout(width="220px"))
        self.pause_button = widgets.Button(description="Pause",
                                           layout=widgets.Layout(width="125px"))
        self.menu_button = widgets.Button(description="Menu", layout=widgets.Layout(width="100px"))
        for button in (self.start_button,self.pause_button,self.menu_button):
            button.style.button_color="#334746"
        self.start_button.on_click(lambda _: self.start_campaign())
        self.pause_button.on_click(lambda _: self.toggle_pause())
        self.menu_button.on_click(lambda _: self.show_menu())
        self.menu_controls = widgets.HBox([self.size_menu,self.start_button])
        self.game_controls = widgets.HBox([self.pause_button,self.menu_button])
        self.up = widgets.Button(description="↑ Up")
        self.left = widgets.Button(description="← Left")
        self.down = widgets.Button(description="↓ Down")
        self.right = widgets.Button(description="→ Right")
        self.scan_button = widgets.Button(description="Scan")
        self.wait_button = widgets.Button(description="Wait")
        self.hint_button = widgets.Button(description="Hint · costly")
        for button,action in ((self.up,lambda:self.move(0,-1)),
                              (self.left,lambda:self.move(-1,0)),
                              (self.down,lambda:self.move(0,1)),
                              (self.right,lambda:self.move(1,0)),
                              (self.scan_button,self.scan_front),
                              (self.wait_button,self.wait_turn),
                              (self.hint_button,self.toggle_hint)):
            button.on_click(lambda _,action=action:action())
            button.style.button_color="#243235"
        self.drive_controls = widgets.VBox([
            widgets.HBox([self.up]),
            widgets.HBox([self.left,self.down,self.right]),
            widgets.HBox([self.scan_button,self.wait_button,self.hint_button])
        ])
        self.title = widgets.HTML()
        self.board = widgets.HTML()
        self.cards = widgets.HTML()
        self.status = widgets.HTML()
        self.challenge_box = widgets.VBox()
        self.ad_box = widgets.VBox()
        self.music_player = widgets.HTML(
            f'<div style="background:#12191b;color:#a7b5b0;padding:7px;font:13px Arial">'
            f'Background music <audio controls loop preload="none" '
            f'src="data:audio/wav;base64,{MUSIC_DATA}" style="height:30px;vertical-align:middle;'
            f'margin-left:12px"></audio></div>')
        self.sound_output = widgets.Output()
        self.ui = widgets.VBox([self.title,self.menu_controls,self.game_controls,self.sound_toggle,
                                self.music_player,self.board,self.challenge_box,self.ad_box,self.drive_controls,
                                self.cards,self.status,self.sound_output])
        self.render()

    def beep(self, frequency):
        if not self.sound_toggle.value:
            return
        samples=[math.sin(2*math.pi*frequency*t/12000)*.15 for t in range(1200)]
        with self.sound_output:
            clear_output(wait=True)
            display(Audio(samples,rate=12000,autoplay=True,normalize=False))

    def show_menu(self):
        self.session += 1
        self.state = "menu"
        self.paused = False
        self.challenge_kind = None
        self.ads_pending = False
        self.render()

    def start_campaign(self):
        self.session += 1
        self.mode = self.size_menu.value
        self.seed = random.SystemRandom().randrange(10**9)
        self.rng.seed(self.seed)
        self.level_number = 1
        self.show_hint = False
        self.load_level()

    def load_level(self):
        if self.level_number > TOTAL_LEVELS:
            self.state = "finished"
            self.render()
            return
        size = level_size(self.mode,self.level_number,self.seed)
        self.floor = generate_floor(size,self.level_number,self.seed+self.level_number*1741)
        self.target = self.rng.choice(("A","B"))
        self.route,self.expanded = bfs_trace(self.floor.start,self.floor.wards[self.target],
                                             size,self.floor.walls)
        self.player_position = self.floor.start
        self.moves = 0
        self.score = 0
        self.energy = max(35,round((len(self.route)-1)*2.8))
        self.start_distance = len(self.route)-1
        self.facing = (1,0)
        self.scanned_cell = self.hazard_state = None
        previous = self.theme_index
        while self.theme_index == previous:
            self.theme_index = self.rng.randint(1,158)
        self.robot_name = "Space Ambulance"
        self.ads_total = 4 + self.level_number
        self.ads_remaining = self.ads_total
        self.ads_pending = True
        self.show_hint = False
        self.hidden_state = "free" if self.rng.random()<self.floor.prior else "blocked"
        self.reports = []
        self.bayes_result = self.hmm_result = self.mdp_result = None
        self.challenge_solved = set()
        self.challenge_kind = None
        self.challenge_result = False
        self.events.clear()
        self.events.append(f"Drive {self.robot_name} to Ward {self.target}. Clear {self.ads_total} signal tabs.")
        self.state = "running"
        self.paused = False
        self.pause_button.description = "Pause"
        self.render()
        self.show_challenge("BFS")

    def toggle_pause(self):
        if self.state not in ("running","challenge","interference","paused"):
            return
        if self.paused:
            self.paused=False
            self.state=self.resume_state
        else:
            self.resume_state=self.state
            self.paused=True
            self.state="paused"
        self.pause_button.description = "Resume" if self.paused else "Pause"
        self.render()

    def front_cell(self):
        return (self.player_position[0]+self.facing[0],
                self.player_position[1]+self.facing[1])

    def valid_cell(self,cell):
        x,y=cell
        return 0<=x<self.floor.size and 0<=y<self.floor.size and cell not in self.floor.walls

    def scan_front(self):
        if self.state!="running" or self.paused:
            return
        cell=self.front_cell()
        if not self.valid_cell(cell):
            self.events.append("Wall ahead. Turn toward a corridor.")
            self.render()
            return
        if self.scanned_cell!=cell or self.hazard_state is None:
            old=self.hidden_state
            self.hidden_state="free" if self.rng.random()<HMM_TRANSITION[old]["free"] else "blocked"
            self.hazard_state=self.hidden_state
            self.scanned_cell=cell
        p_clear=self.floor.sensitivity if self.hazard_state=="free" else self.floor.false_alarm
        report="CLEAR" if self.rng.random()<p_clear else "ALERT"
        self.reports.append(report)
        p_free,first,other=bayes_free(report,self.floor.prior,
                                      self.floor.sensitivity,self.floor.false_alarm)
        self.bayes_result=(report,p_free,first,other)
        if len(self.reports)>=3:
            self.hmm_result=viterbi(self.reports[-3:],self.floor.prior,
                                    self.floor.sensitivity,self.floor.false_alarm)
        self.mdp_result=mdp_decision(p_free)
        self.energy-=1
        self.events.append(f"Scan {cell}: {report}; advice {self.mdp_result[0].title()}.")
        self.check_energy()
        self.render()

    def move(self,dx,dy):
        if self.state!="running" or self.paused:
            return
        if (dx,dy)!=self.facing:
            self.facing=(dx,dy)
            self.scanned_cell=self.hazard_state=None
        cell=self.front_cell()
        if not self.valid_cell(cell):
            self.energy-=1
            self.score=max(0,self.score-2)
            self.events.append("Wall ahead. Choose another direction.")
            self.check_energy()
            self.render()
            return
        if cell==self.floor.wards[self.target] and "MDP" not in self.challenge_solved:
            self.show_challenge("MDP",pending=(dx,dy))
            return
        if self.scanned_cell!=cell or self.hazard_state is None:
            self.scan_front()
        if self.state!="running":
            return
        risky=self.mdp_result is not None and self.mdp_result[0]=="PAUSE"
        self.player_position=cell
        self.moves+=1
        self.score+=2 if risky else 5
        self.energy-=2 if risky else 1
        self.scanned_cell=self.hazard_state=None
        self.route,self.expanded=bfs_trace(cell,self.floor.wards[self.target],
                                            self.floor.size,self.floor.walls)
        if cell==self.floor.wards[self.target]:
            self.state="level_complete"
            self.events.append(f"Ward {self.target} reached!")
            self.beep(800)
            session=self.session
            asyncio.get_running_loop().create_task(self.next_level_after_delay(session))
        else:
            self.events.append((f"Moved to {cell} despite warning; extra energy used."
                                if risky else f"Moved to {cell}."))
            self.check_energy()
        self.render()
        self.trigger_challenge()

    async def next_level_after_delay(self,session):
        await asyncio.sleep(1.2)
        if session!=self.session or self.state!="level_complete":
            return
        self.level_number+=1
        self.load_level()

    def wait_turn(self):
        if self.state!="running" or self.paused:
            return
        self.energy-=1
        self.scanned_cell=self.hazard_state=None
        self.events.append("Waited one turn; the corridor may change.")
        self.check_energy()
        self.render()

    def toggle_hint(self):
        if self.state!="running":
            return
        if not self.show_hint:
            cost=max(18,round(max(35,self.start_distance*2.8)*.5))
            if self.energy<=cost:
                self.events.append(f"Route hint needs {cost} Energy; current {self.energy}.")
                self.render()
                return
            self.energy-=cost
            self.events.append(f"Route hint on. Signal boost cost {cost} Energy.")
        else:
            self.events.append("Route hint off.")
        self.show_hint=not self.show_hint
        self.render()

    def show_challenge(self,kind,pending=None):
        if self.state!="running" or kind in self.challenge_solved:
            return
        self.state="challenge"
        self.challenge_kind=kind
        self.challenge_pending=pending
        self.challenge_result=False
        self.challenge_message="A correct answer moves the robot one step."
        if kind=="BFS":
            distance=len(self.route)-1
            self.challenge_prompt=(f"BFS explored {self.expanded} cells. "
                                   f"How many moves does the highlighted shortest route to Ward {self.target} have?")
            self.challenge_answer=distance
            self.challenge_options=[max(1,distance-2),distance,distance+2]
            self.challenge_explanation=(f"The route has {len(self.route)} cells, "
                                        f"so it has {distance} moves (edges).")
        elif kind=="Bayes":
            report,p,first,other=self.bayes_result
            self.challenge_prompt=(f"Report {report}; prior P(free)={self.floor.prior:.0%}; "
                                   f"P(CLEAR|free)={self.floor.sensitivity:.0%}; "
                                   f"P(CLEAR|blocked)={self.floor.false_alarm:.0%}. "
                                   "What is P(free|report), rounded to a whole percent?")
            answer=round(p*100)
            self.challenge_answer=answer
            self.challenge_options=list(dict.fromkeys((answer,round(self.floor.prior*100),round((1-p)*100))))
            while len(self.challenge_options)<3:
                self.challenge_options.append((answer+13*len(self.challenge_options))%100)
            self.challenge_explanation=f"Bayes: {first:.3f}/({first:.3f}+{other:.3f})={p:.1%}."
        elif kind=="HMM":
            path,trace=self.hmm_result
            self.challenge_prompt=("Reports: "+" → ".join(self.reports[-3:])+
                                   f". Initial P(free)={self.floor.prior:.0%}; "
                                   f"P(CLEAR|free)={self.floor.sensitivity:.0%}; "
                                   f"P(CLEAR|blocked)={self.floor.false_alarm:.0%}. "
                                   "Transitions F→F 80%, F→B 20%, B→F 30%, B→B 70%. "
                                   "What is the most likely final hidden state?")
            self.challenge_answer=path[-1].upper()
            self.challenge_options=["FREE","BLOCKED"]
            self.challenge_explanation=("Viterbi: "+" → ".join(x.upper() for x in path)+
                                        f". Final F={trace[-1]['free']:.3f}, B={trace[-1]['blocked']:.3f}.")
        else:
            probability=self.bayes_result[1] if self.bayes_result else self.floor.prior
            action,q=mdp_decision(probability)
            self.challenge_prompt=(f"P(free)={probability:.1%}. Continue: +10 free / −20 blocked. "
                                   "Pause: −2 free / +2 blocked. Which action has higher expected reward?")
            self.challenge_answer=action
            self.challenge_options=["CONTINUE","PAUSE"]
            self.challenge_explanation=(f"Q(Continue)={q['CONTINUE']:+.2f}; "
                                        f"Q(Pause)={q['PAUSE']:+.2f}. Choose {action}.")
        self.rng.shuffle(self.challenge_options)
        self.render()

    def answer_challenge(self,answer):
        if self.state!="challenge":
            return
        if answer!=self.challenge_answer:
            self.challenge_message="Not quite. Try another answer."
        else:
            self.challenge_solved.add(self.challenge_kind)
            self.score+=15
            self.challenge_result=True
            self.challenge_message=self.challenge_explanation
            self.beep(650)
        self.render()

    def continue_challenge(self):
        pending=self.challenge_pending
        self.challenge_kind=None
        self.challenge_pending=None
        self.state="running"
        self.render()
        if pending is None and len(self.route)>1:
            nxt=self.route[1]
            pending=(nxt[0]-self.player_position[0],nxt[1]-self.player_position[1])
        if pending is not None:
            self.move(*pending)
        if self.state=="running" and self.ads_pending:
            self.show_ads()

    def show_ads(self):
        if not self.ads_pending or self.state!="running":
            return
        self.ads_pending=False
        self.state="interference"
        self.render()

    def close_ad(self):
        if self.state!="interference" or self.ads_remaining<=0:
            return
        self.ads_remaining-=1
        if self.ads_remaining==0:
            self.state="running"
            self.events.append("Signal clear. Ambulance controls restored.")
        self.render()

    def trigger_challenge(self):
        if self.state!="running":
            return
        if ("Bayes" not in self.challenge_solved and self.moves>=max(2,self.start_distance//3)
            and self.bayes_result):
            self.show_challenge("Bayes")
        elif ("HMM" not in self.challenge_solved and self.moves>=max(4,2*self.start_distance//3)
              and self.hmm_result):
            self.show_challenge("HMM")

    def check_energy(self):
        if self.energy<=0 and self.state=="running":
            self.state="failed"
            self.events.append("Battery empty. Return to menu to try again.")

    def map_html(self):
        f=self.floor
        p=theme_colors(self.theme_index)
        family=p["family"]
        view=min(f.size,19)
        px,py=self.player_position
        left=max(0,min(f.size-view,px-view//2))
        top=max(0,min(f.size-view,py-view//2))
        tile=38 if f.size<=20 else 29
        route=set(self.route) if (self.show_hint or
              (self.state=="challenge" and self.challenge_kind=="BFS")) else set()
        items=[]
        for y in range(top,top+view):
            for x in range(left,left+view):
                cell=(x,y)
                wall=cell in f.walls
                color=p["wall"] if wall else p["floor"]
                background=(f"repeating-linear-gradient(90deg,transparent 0px,transparent 8px,#00000008 9px),{color}"
                            if family==1 else f"linear-gradient(145deg,#ffffff30,transparent),{color}"
                            if family==2 else color)
                symbol=(self.ROBOT_SYMBOLS[self.robot_name] if cell==(px,py) else
                        "A" if cell==f.wards["A"] else "B" if cell==f.wards["B"] else
                        "S" if cell==f.start else "•" if cell in route else "")
                fg=p["accent"] if cell in route or cell==(px,py) else p["ink"]
                if cell==(px,py):
                    symbol=(f'<span style="display:flex;align-items:center;justify-content:center;'
                            f'width:{tile+7}px;height:{tile+7}px;background:{p["highlight"]};'
                            f'border:2px solid {p["accent"]};border-radius:50%;'
                            f'animation:botbob .5s ease-in-out infinite alternate;'
                            f'color:#a86d69">{symbol}</span>')
                items.append(f'<div style="width:{tile}px;height:{tile}px;display:flex;align-items:center;'
                             f'justify-content:center;background:{background};border:1px solid {p["line"]};'
                             f'color:{fg};font:bold {max(12,tile//2)}px sans-serif">{symbol}</div>')
        return ('<style>@keyframes botbob{from{transform:translateY(1px)}to{transform:translateY(-3px)}}</style>'
                f'<div style="display:grid;grid-template-columns:repeat({view},{tile+2}px);width:fit-content;'
                f'border:8px solid {p["card"]}">' + ''.join(items) + '</div>')

    def render(self):
        self.menu_controls.layout.display = "flex" if self.state=="menu" else "none"
        self.game_controls.layout.display = "none" if self.state=="menu" else "flex"
        self.drive_controls.layout.display = "flex" if self.state=="running" else "none"
        self.challenge_box.layout.display = "flex" if self.state=="challenge" else "none"
        self.ad_box.layout.display = "flex" if self.state=="interference" else "none"
        if self.state=="interference":
            closed=self.ads_total-self.ads_remaining
            title=widgets.HTML(f'<div style="background:#1c2729;color:#e1e8e4;'
                               f'border:2px solid #6f9f91;padding:14px;max-width:700px">'
                               f'<b>SIGNAL INTERFERENCE</b><br>Close every incoming tab · '
                               f'{closed}/{self.ads_total} cleared</div>')
            tabs=[]
            for i in range(min(5,self.ads_remaining)):
                button=widgets.Button(description=f'Tab {closed+i+1:02d} ×',
                                      layout=widgets.Layout(width='115px'))
                button.style.button_color='#243235'
                button.on_click(lambda _:self.close_ad())
                tabs.append(button)
            self.ad_box.children=(title,widgets.HBox(tabs))
        else:
            self.ad_box.children=()
        if self.state=="challenge":
            heading=widgets.HTML(
                f'<div style="background:#1c2729;color:#e1e8e4;border:2px solid #6f9f91;'
                f'padding:14px;max-width:700px;font:14px Arial">'
                f'<b>{self.challenge_kind.upper()} CHALLENGE</b><br>{self.challenge_prompt}<br><br>'
                f'<span style="color:#a7b5b0">{self.challenge_message}</span></div>')
            if self.challenge_result:
                button=widgets.Button(description="Continue · move one step",
                                      layout=widgets.Layout(width="320px"))
                button.style.button_color="#4e766c"
                button.on_click(lambda _:self.continue_challenge())
                self.challenge_box.children=(heading,button)
            else:
                answers=[]
                for option in self.challenge_options:
                    suffix="%" if self.challenge_kind=="Bayes" else " moves" if self.challenge_kind=="BFS" else ""
                    button=widgets.Button(description=f"{option}{suffix}",
                                          layout=widgets.Layout(width="320px"))
                    button.style.button_color="#243235"
                    button.on_click(lambda _,option=option:self.answer_challenge(option))
                    answers.append(button)
                self.challenge_box.children=(heading,*answers)
        else:
            self.challenge_box.children=()
        if self.state=="menu":
            self.title.value='<h2 style="font-family:Arial;color:#e1e8e4;background:#12191b;padding:14px">Orbital Ambulance</h2>'
            self.board.value='<div style="background:#1c2729;color:#e1e8e4;padding:14px">Choose a map size. Drive the space ambulance with the on-screen arrows.</div>'
            self.cards.value='<div style="background:#12191b;color:#a7b5b0;padding:12px">BFS · Bayes · HMM · MDP all run in Python.</div>'
            self.status.value='<div style="background:#12191b;color:#a7b5b0;padding:12px">Random size changes on every level and never repeats immediately.</div>'
            return
        if self.state=="finished":
            self.title.value='<h2 style="color:#e1e8e4;background:#12191b;padding:14px">Campaign complete</h2>'
            self.board.value='<p style="color:#e1e8e4;background:#1c2729;padding:14px">All 30 deliveries finished. Click Menu to play again.</p>'
            self.cards.value=''
            self.status.value=''
            return
        f=self.floor
        p=theme_colors(self.theme_index)
        self.title.value=(f'<div style="font: bold 22px Arial;background:{p["background"]};'
                          f'color:{p["ink"]};padding:10px">'
                          f'Level {self.level_number:02d}/30 · {f.size}×{f.size} · Ward {self.target}'
                          f'<span style="font:13px Arial;color:{p["muted"]};margin-left:16px">'
                          f'{self.robot_name} · {STYLE_NAMES[p["family"]]} · Look {self.theme_index}/158</span></div>')
        self.board.value=self.map_html()
        bfs_text=(f'From {self.player_position} to Ward {self.target}. '
                  f'Queue expanded {self.expanded} cells; {len(self.route)-1} moves remain. '
                  f'Hint is {"on" if self.show_hint else "off"}.')
        if self.state=="challenge" and self.challenge_kind=="BFS":
            bfs_text=f'Queue expanded {self.expanded} cells. Count the highlighted route.'
        if self.bayes_result:
            report,prob,first,other=self.bayes_result
            bayes_text=f'{report}; {first:.3f}/({first:.3f}+{other:.3f})={prob:.1%} free.'
        else:
            bayes_text='Waiting for first sensor report.'
        if self.state=="challenge" and self.challenge_kind=="Bayes":
            bayes_text=f'Report {self.bayes_result[0]}. Calculate P(free|report).'
        if self.hmm_result:
            states,trace=self.hmm_result
            hmm_text=('Reports: '+' → '.join(self.reports[-3:])+'; Viterbi: '
                      +' → '.join(x.title() for x in states)
                      +f'; final F={trace[-1]["free"]:.3f}, B={trace[-1]["blocked"]:.3f}.')
        else:
            hmm_text=f'Collecting three reports: {len(self.reports)}/3.'
        if self.state=="challenge" and self.challenge_kind=="HMM":
            hmm_text='Reports: '+' → '.join(self.reports[-3:])+'. Find the final hidden state.'
        if self.mdp_result:
            action,q=self.mdp_result
            mdp_text=(f'Q(Continue)={q["CONTINUE"]:+.2f}; Q(Pause)={q["PAUSE"]:+.2f}. '
                      f'Advice: {action.title()}; you choose.')
        else:
            mdp_text='Waiting for Bayes probability.'
        if self.state=="challenge" and self.challenge_kind=="MDP":
            mdp_text='Compare expected rewards for Continue and Pause.'
        cards=[('Route · BFS',bfs_text),('Sensor · Bayes',bayes_text),
               ('Pattern · HMM',hmm_text),('Action · MDP',mdp_text)]
        self.cards.value=''.join(
            f'<div style="background:#1c2729;border:1px solid #344245;padding:10px;margin:5px 0;max-width:700px;'
            f'font:14px Arial;color:#e1e8e4"><b>{title}</b><br>{text}</div>' for title,text in cards)
        state=(f'Solve {self.challenge_kind} challenge' if self.state=='challenge' else
               f'Close {self.ads_remaining} signal tabs' if self.state=='interference' else
               'Paused' if self.paused else 'Battery empty' if self.state=='failed' else
               'Delivery complete — next level starts automatically' if self.state=='level_complete' else 'Your turn')
        progress=100*(1-(len(self.route)-1)/max(1,self.start_distance))
        x,y=self.player_position
        self.status.value=(f'<p style="font:14px Arial;color:#e1e8e4;background:#12191b;padding:8px">'
                           f'{state} · Robot at ({x},{y}) · Ward {self.target} · '
                           f'Energy {self.energy} · Score {self.score} · Moves {self.moves}. '
                           +' '.join(self.events)+'</p>'
                           f'<div style="height:7px;max-width:700px;background:#344245">'
                           f'<div style="height:7px;width:{progress:.1f}%;background:{p["accent"]}"></div></div>')

game=ColabCampaign()
display(game.ui)

## Method notes for the presentation

- **BFS:** input is the generated grid and target ward. A queue explores neighbors until it reaches the ward. The result is a shortest route.
- **Bayes:** input is one CLEAR or ALERT report plus prior and sensor accuracy. The posterior is shown as numerator divided by total probability.
- **HMM/Viterbi:** input is the latest three reports, two hidden states and fixed transition probabilities. The result is the most likely hidden state sequence.
- **MDP:** input is the Bayes probability, two corridor states, two actions and fixed rewards. The result is the action with higher one-step expected reward.

Topic 14 formally requires BFS, Bayes and MDP. HMM is included as an extra course demonstration. All assumptions are simulated; no patient data or trained model is used. This project code was prepared with AI assistance and verified with the checks below.

In [ ]:
# Core checks, including large maps and random size progression.
for size in (5,20,100):
    for level in (1,15,30):
        floor=generate_floor(size,level,1000+level)
        assert all(bfs(floor.start,goal,size,floor.walls) for goal in floor.wards.values())
sequence=[level_size("Random",i,123) for i in range(1,31)]
assert all(a!=b for a,b in zip(sequence,sequence[1:]))
assert len({theme_colors(i)["accent"] for i in range(1,159)})==158
p,_,_=bayes_free("CLEAR",.75,.9,.2)
assert abs(p-.675/.725)<1e-10
assert mdp_decision(p)[0]=="CONTINUE"
assert viterbi(("CLEAR","CLEAR","CLEAR"),.75,.9,.2)[0]==("free","free","free")
print("BFS, Bayes, HMM, MDP, 30-level generator and 158 looks: OK")